In [ ]:
import csv
import pandas as pd
import re

In [ ]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df_col

In [ ]:
def split_multiple_choices(data, column, index, sep = '|'):
    """
    split and explode column with multiple value

    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_split = data.copy()
    df_split[column]= df_split.apply(lambda row: row[column].replace(";","|") ,1 )
    df_split[column] = df_split[column].str.split(sep)
    df_explode = df_split.explode(column)
    gb_data = df_explode.groupby([column]).agg(nb = (index, "size")).sort_values("nb", ascending=False).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100
    
    return df_explode, gb_data
    

In [ ]:
def grouped_question(data, column, index = "q45_clé"):
    """


    data = a panda dataframe
    column = name of column which we want to split.
    index = column corresponding to id of rows
    sep = by default '|'. Character use to separate values.
    """
    df_tmp = data.copy()
    gb_data = df_tmp.groupby([column]).agg(nb = (index, "size")).reset_index()
    gb_data["total"] = data[index].nunique()
    gb_data["freq"] = gb_data.nb/gb_data.total*100
    gb_data["total_freq"] = gb_data.total/gb_data.total*100

    return gb_data

In [ ]:
def tab_croise(data, x, y, regroup_y = True, khideux = False) :
    """
    x = variable en ligne
    y = variable en colonne
    """
    if regroup_y == True:

        data.loc[data[y].str.lower().str.contains("oui"), f"{y}_rec"] = "Oui"
        data.loc[data[y].str.lower().str.contains("non"), f"{y}_rec"] = "Non"

        cross_tab = pd.crosstab(data[x], data[f"{y}_rec"], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[f"{y}_rec"], margins = False)
    else:
        cross_tab = pd.crosstab(data[x], data[y], margins = True, margins_name="Total", normalize=False)
        cross_tab0 = pd.crosstab(data[x], data[y], margins = False, normalize=False)

    if khideux == True:
        print(scipy.stats.chi2_contingency(cross_tab0))
        st_chi2, st_p, st_dof, st_exp = scipy.stats.chi2_contingency(cross_tab0)
        chi2 = pd.DataFrame(data={"stats":["chi2","df","p-value"], "values":[st_chi2, st_dof, st_p]})
        cross_tab = pd.concat([cross_tab, chi2])
    else:
        pass

    return cross_tab.fillna("").reset_index(), cross_tab0

# Recodage variable q25_statut

On recode les personnes qui ont répondu Autre à la question : "Quel est votre statut ?"
Je propose également de faire une autre variable : "Doctorant/non doctorant"

In [ ]:


df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_statut_rec"] = "Chercheur"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_statut_rec"] = "Doctorant"
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_statut_rec"] = "Personnel d'appui à la recherche"

df0.loc[df0.q25_statut=="Doctorant·e", "q25_statut_rec"] = "Doctorant"
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_statut_rec"] = "Chercheur"
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_statut_rec"] = "Personnel d'appui à la recherche"
df0.q25_statut_rec.value_counts()


dic_new_var = {'name':'110. statut_rec', 
               'label':'q25_statut_rec', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_statut_rec.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [ ]:
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"chercheur")), "q25_doctorant"] = False
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"doctorante|docteur")), "q25_doctorant"] = True
df0.loc[(df0.q25_statut=="Autre")&(df0.q25_1_statut_autre.str.lower().str.contains(r"ing")), "q25_doctorant"] = False

df0.loc[df0.q25_statut=="Doctorant·e", "q25_doctorant"] = True
df0.loc[df0.q25_statut=="Enseignant·e chercheur·e", "q25_doctorant"] = False
df0.loc[df0.q25_statut=="Personnel de soutien à la recherche", "q25_doctorant"] = False
df0.q25_doctorant.value_counts()

dic_new_var = {'name':'110. doctorant', 
               'label':'q25_doctorant', 
               'group':'12_descript_repondant', 
               'personal_data':False, 
               'type':'simple_nominal', 
               'opened_question':False,
               'type_panda':df0.q25_doctorant.dtypes,
               'no_question':25,
               'question':df_col.question.loc[df_col.label=="q25_statut"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)


In [ ]:
df0.loc[df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Non"
df0.loc[~df0.q3_octavi_depot.str.contains("Non|Je ne connais pas"), "q3_octavi_rec"] = "Oui"


In [ ]:
dic_new_var = {'name':'3. octavi_depot_rec',
               'label':'q3_octavi_rec',
               'group':'2_bib_num',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df0.q3_octavi_rec.dtypes,
               'no_question':3,
               'question':df_col.question.loc[df_col.label=="q3_octavi_depot"].iloc[0]
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [ ]:
df_col.to_csv("../le_questionnaire/dico_variable.csv", sep=",", index= False)

# Production et analyse des données


## Recodage "autres langages"

In [ ]:
rec_other_lang ={}
for x in df0.q5_1_other_lang.loc[~df0.q5_1_other_lang.isna()]:
    rec_other_lang[x] = x.lower().replace(", ","|")

rec_other_lang

In [ ]:
rec_other_lang = {'MaxQda, EndNotes, Zotero, ': 'maxqda|endnotes|zotero',
 '\u200b-': 'aucun',
 'Matlab': 'matlab',
 'Zotero': 'zotero',
 'LibreOffice Calc': 'libreoffice calc',
 'csv, google sheets': 'csv|google sheets',
 'Jamovi, Iramuteq, Qualcoder': 'jamovi|iramuteq|qualcoder',
 'Nvivo': 'nvivo',
 'php, javascript, d3.js': 'php|javascript|d3.js',
 'Jamovi': 'jamovi',
 'JAMOVI ': 'jamovi',
 'c#': 'c#',
 'SPSS, Statistica, Jamovi, JASP': 'spss|statistica|jamovi|jasp',
 'Nvivo, IA': 'nvivo|ia',
 'Nvivo et Atlas.ti': 'nvivo|atlas.ti',
 'SPSS': 'spss',
 'OCaml, C, Bash, PHP': 'ocaml|c/c++|bash|php',
 'CSS, html, xml': 'css|html|xml',
 'LaTeX': 'latex',
 'NVivo': 'nvivo',
 'MatLab': 'matlab',
 "QGIS. J'aimerais utiliser Python, mais il faudrait que je sois formée pour en avoir les bases. J'ai déjà suivi la formation de base de R proposée à P8, qui était bien : reste maintenant à se l'approprier !": "qgis",
 'Access': 'access',
 'File maker pro': 'file maker pro',
 'LibreOffice Calc, Limesurvey, Grist, ActiveTigger, Neo4j, Gephi': 'libreoffice calc|limesurvey|grist|activetigger|neo4j|gephi',
 'Jamovi, SPSS': 'jamovi|spss',
 'C/C++ Cuda Caml Racket Ludii GDL ASP': 'c/c++|cuda|caml|racket|ludii|gdl|asp',
 'R mais avec Iramutex': 'iramuteq',
 'jamovi': 'jamovi',
 'QSR NVivo': 'nvivo',
 'Mplus, SPSS..': 'mplus|spss'}



df0["q5_1_other_lang_rec"]= df0.q5_1_other_lang.map(rec_other_lang.get)

# on enregistre les nouvelles variables dans le dataframe original
#with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
#    df0.to_csv(file_out, sep=",", index= False)


In [ ]:
dic_new_var = {'name':'14. other_lang',
               'label':'q5_1_other_lang_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_1_other_lang_rec.dtypes,
               'no_question':14,
               'question':df_col.question.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_1_other_lang"].iloc[0],
               'comment':"Recodage de la question 14. mise en basse casse des logiciels cités."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col

In [ ]:
list_other_logi = [x for x in df_exp.q5_1_other_lang_rec.unique()]
print(list_other_logi)

In [ ]:
family_logi = {'maxqda':'CAQDAS', 
               'endnotes':'gestion biblio', 
               'zotero':'gestion biblio', 
               'aucun':'aucun', 
               'matlab':'script et programmation', 
               'libreoffice calc':'feuille de calcul', 
               'csv':'feuille de calcul', 
               'google sheets':'feuille de calcul',
               'jamovi':'logiciels de stats',
               'iramuteq':'analyse de texte', 
               'qualcoder':'CAQDAS',
               'nvivo':'CAQDAS',
               'php':'développement web', 
               'javascript':'développement web',
               'd3.js':'développement web', 
               'c#':'script et programmation', 
               'spss':'logiciels de stats',
               'statistica':'logiciels de stats',
               'jasp':'logiciels de stats',
               'ia':'ia', 
               'atlas.ti':'CAQDAS',
               'ocaml':'script et programmation', 
               'c/c++':'script et programmation',
               'bash':'script et programmation',
               'css':'développement web',
               'html':'développement web', 
               'xml':'développement web',
               'latex':'script et programmation',
               'qgis':'cartographie',
               'access':'gestion de base de donnée',
               'file maker pro':'gestion de base de donnée',
               'limesurvey':'logiciel d\'enquête statistique',
               'grist':'gestion de base de donnée',
               'activetigger':'analyse de texte',
               'neo4j':'gestion de base de donnée',
               'gephi':'analyse de réseau',
               'cuda':'script et programmation', 
               'caml':'script et programmation', 
               'racket':'script et programmation', 
               'ludii':'création de jeu',
               'gdl':'logiciels de stats',
               'asp':'développement web',
               'mplus': 'logiciels de stats',
               'excel': 'feuille de calcul',
               'python': 'script et programmation',
               'r':'script et programmation',
               'stata':'logiciels de stats',
               'julia': 'script et programmation',
               'sas': 'logiciels de stats'
              }

In [ ]:
df01 = df0.copy()

dict_logiciel_traitement = {}
for row in df0.q45_clé:
    logiciel_traitement = []
    for x in df_col.label.loc[(df_col.question_family=="q5")& (~df_col.label.str.contains("q5_1"))]:
        if df01[x].loc[df01.q45_clé==row].values == "Oui":
            logiciel_traitement.append(x.lower().replace("q5_",""))
    if len(logiciel_traitement)>0:
        dict_logiciel_traitement[row]= '|'.join(logiciel_traitement)
df01["q5_logiciel_traitement_rec"] = df01.q45_clé.map(dict_logiciel_traitement)



In [ ]:

df_exp, gb_data = split_multiple_choices(df01.loc[~df01.q5_logiciel_traitement_rec.isna()], column='q5_logiciel_traitement_rec', index="q45_clé", sep = '|')
df_exp1, gb_data1 = split_multiple_choices(df01.loc[~df01.q5_1_other_lang_rec.isna()], column='q5_1_other_lang_rec', index="q45_clé", sep = '|')

df_exp2 = pd.concat([df_exp[["q45_clé", 'q5_logiciel_traitement_rec']], df_exp1[["q45_clé","q5_1_other_lang_rec"]].rename(columns={"q5_1_other_lang_rec":'q5_logiciel_traitement_rec'})])
df_exp2["q5_family_logi"]= df_exp2.q5_logiciel_traitement_rec.map(family_logi.get)
df_exp3 = df_exp2.loc[~df_exp2.q5_logiciel_traitement_rec.isin(["aucun","autre"])]
df_exp3

In [ ]:
df01 = df0.copy()

dict_logiciel_traitement2 = {}
dict_logiciel_fam = {}
for row in df_exp3.q45_clé:
    
    dftmp = df_exp3.loc[df_exp3.q45_clé==row]
    logiciel_traitement2 = [x for x in dftmp.q5_logiciel_traitement_rec.unique()]
    family_logi = [x for x in dftmp.q5_family_logi.unique()]
    dict_logiciel_traitement2[row] = "|".join(logiciel_traitement2)
    dict_logiciel_fam[row] = "|".join(family_logi)


In [ ]:
df0["q5_logiciel_traitement_rec"]= df0.q45_clé.map(dict_logiciel_traitement2.get)
df0["q5_family_logi"] = df0.q45_clé.map(dict_logiciel_fam.get)


# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

#On enregistre à côté la liste des familles de logiciel. Permettra de "mapper" plus facilement le nom des logiciels avec leurs familles pour les traitements ultérieurs
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/type_of_software.csv", "w") as file_out:
    df_exp2.drop(columns=["q45_clé"]).drop_duplicates().to_csv(file_out, sep=",", index = False)

In [ ]:
dic_new_var = {'name':'13.1 logiciel de traitment',
               'label':'q5_logiciel_traitement_rec',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_logiciel_traitement_rec.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Regroupe les questions 5 et 6 sur les logiciels de traitement"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

dic_new_var = {'name':'13.2 familles de logiciel',
               'label':'q5_family_logi',
               'group':'4_logiciel_traitement',
               'personal_data':False,
               'type':'multiple_choice',
               'opened_question':False,
               'type_panda':df0.q5_family_logi.dtypes,
               'no_question':5,
               'question':df_col.question.loc[df_col.label=="q5_r"].iloc[0],
               "question_family":df_col.question_family.loc[df_col.label=="q5_r"].iloc[0],
               'comment':"Classe les logiciels par famille"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1


## Développement et dépôt de codes ou de logiciels

In [ ]:
rec_code={
    "Mes activités de recherche consist à développer des outils spécialisés pour le traitement de la parole et l'analyse des médias":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "des apps web": 
    {"what":"applications web","group":"programmes et logiciels variés", "how":""},
    
    "Des logiciels variés, allant du materiel d'expérience à des productions directes à être diffusées.":
    {"what":"outils d'expériences", "group":"collecte de données|programmes et logiciels variés", "how":""},
    
    "Questionnaires et programmes d'expérience sur Psytoolkit":
    {"what":"outils d'expérience","group":"collecte de données", "how":"psychtoolkit"},
    
    "Beaucoup d'outils cf. https://samszo.jardindesconnaissances.fr/HDR/cv/cv.html#sec-item299386":
     {"what":"", "group":"programmes et logiciels variés", "how":""},
    
    "Les Commades Stata":
    {"what":"des commandes stata", "group":"traitement et analyse de données", "how":"Stata"},
    
    "Plugin de moteur temps réel pour la XR.":
    {"what":"plugins pour réalité virtuelle", "group":"programmes et logiciels variés", "how":""},
    
    "Je ne l'ai pas fait moi-même mais à grenoble je travaille avec les ingénieurs de la PUD pour fabriqueur un outil semi automatique de pseudonymisation.":
    {"what":"outils de pseudonymisation", "group": "traitement et analyse de données","how":""},
    
    "Une librairie Julia pour du traitement de réseaux.":
    {"what":"analyse de réseau", "group":"traitement et analyse de données", "how":"Julia"},
    
    "plateformes, programmes logiciels, œuvres":
    {"what":"programmes et logiciels variés", "group":"programmes et logiciels variés", "how":""},
    
    "Logiciels de synthèse et traitement du son":
    {"what":"traitement du son","group":"traitement et analyse de données", "how":""},
    
    "Outil de transcription et d'annotation de données": 
    {"what":"transcription et annotation", "group":"traitement et analyse de données", "how":""},
    
    "https://pablo.rauzy.name/software.html":
    {"what":"", "group":"programmes et logiciels variés","how":""},
    
    "création de site internet pour valorisation et diffusion des activités de recherche, productions des étudiants, etc": 
    {"what":"sites web", "group": "diffusion", "how":""},
    
    "Implémentation d's ou algorithme de démonstration":
    {"what":"implémentation", "group": "programmes et logiciels variés", "how":""},
    
    "jeu sérieux sur tablette": 
    {"what":"jeu sérieux", "group":"collecte de données", "how":""},
    
    "Développez les scripts/méthodologies pour traiter mes données":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""},
    
    "Plateforme PatriMaths, sémathèque : https://sematheque.ahp-numerique.fr":
    {"what":"plateforme","group":"diffusion","how":""},
    
    "Dans le cadre du consortium Projets Time Machine, plusieurs logiciels/codes spécialisés sont des logiciels: d'analyse morphologique (MorphAL = Analyse morphologique) qui a énté intégrée comme un plug-in dans QGis ; Amado-on-line (logiciel de visualisation des matrices graphiques le suivant de Bertin)":
    {"what":"analyse morphologique", "group":"traitement et analyse de données", "how":"qgis"},
    
    "prédication de C02 dans Paris":
    {"what":" prédiction", "group":"traitement et analyse de données", "how":""},
    "Scripts Python. En ce moment : prestation avec le CERES pour met à jour Digipower Academy, en partenariat avec l'association suisse Personaldata.io":
    {"what":"applications web", "group":"programmes et logiciels variés","how":"python"},
    
    "jouer aux jeux à 1, à 2 et joueurs et + selon les jeux (ou problèmes considérés)":
    {"what":"jeux", "group":"collecte de données", "how":""},
    
    "Des logiciels d'Interaction Humain-Machine transportante des appareils pétéraux d'interactions (recueil de données de protocoles d'interaction)":
    {"what":"logiciels d'interaction humain-machine", "group":"collecte de données", "how":""},
    
    "Des cahiers Jupyter pour l'analyse des données de recherche":
    {"what":"notebook", "group":"traitement et analyse de données","how":"jupyter"},
    
    "Nous avons avec l'INA une IA (gpt-oss-120) pour thématiser de gros cormus de vidéos":
    {"what":"classification automatique", "group":"traitement et analyse de données", "how":"IA générative"},
    
    "une série d'outil de modélisation à base d'approche profond lerning":
    {"what":"modélisation",  "group":"traitement et analyse de données", "how":"deep learning"},
    
    "Dans un projet de recherche je participe à participer, nous faisons réaliser un HTR de formulaires manuscrits.":
    {"what":"reconnaissance de caractère", "group":"traitement et analyse de données","how":""},
    
    "des code pour les expériences à l'aide d'outils type PsychToolkit":
    {"what":"outils d'expérience", "group":"collecte de données", "how":"psychtoolkit"},
    "TEI/XML adapté à une correspondance du xviiie siècle":
    {"what":"transcription et annotation", "group":"traitement et analyse de données","how":"TEI/XML"},
    
    "Base de de donation TOFLLIT18 (issu d'un projet ANR)":
    {"what":"base de données", "group":"collecte de données", "how":""},
    
    "des scripts R pour du traitement d'analyse de despageses":
    {"what":"scripts", "group":"traitement et analyse de données", "how":""}
}

In [ ]:
list_exemple_code = [x for x in df0.q6_1_dev_exemple.loc[~df0.q6_1_dev_exemple.isna()]]
new_rec_code = {}
for n, x in enumerate(rec_code):
    new_key = list_exemple_code[n]
    new_rec_code[new_key]= rec_code[x]

new_rec_code


In [ ]:
list_row= []
for x in new_rec_code:

    id_row = df0.q45_clé.loc[df0.q6_1_dev_exemple==x].values

    if len(id_row) > 0:
        exemple = new_rec_code[x]
        dict_row = {"q45_clé":id_row[0],
              "q6_1_dev_what":exemple["what"].lower(),
              "q6_1_dev_group":exemple["group"].lower(),
              "q6_1_dev_how": exemple["how"].lower()}
        list_row.append(dict_row)

dftp = pd.DataFrame.from_dict(list_row)
dftp


In [ ]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_what',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_what.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Recodage de la question 16. Restitue le type d'objet développé : traitement du son, jeu, base de donnée, site web"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_group',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_group.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Classe les exemples de logiciels développés dans des groupes : outils de traitement et d'analyse, outils de diffusion, collecte de données, etc."
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
dic_new_var = {'name':'16. dev_exemple',
               'label':'q6_1_dev_how',
               'group':'5_developpement',
               'personal_data':False,
               'type':'simple_nominal',
               'opened_question':False,
               'type_panda':df01.q6_1_dev_how.dtypes,
               'no_question':16,
               'question':df_col.question.loc[df_col.label=="q6_1_dev_exemple"].iloc[0],
               'comment':"Précise le langage ou l'outil de programmation utilisé"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col1,new_variable], ignore_index = True)
df_col1

In [ ]:
df_col.to_csv("../../le_questionnaire/dico_variable.csv", sep=",", index= False)

## Profils répondants

### Recodage autres lieux de travail

In [ ]:
for x in df0.q26_1_autre_env.loc[~df0.q26_1_autre_env.isna()]:
    print(x)

In [87]:
dict_autre_env ={
    "Bibliothèque" :"Bibliothèque",
"Dans les locaux de mes partenaires de recherche":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques":"Bibliothèque",
"Dans le métro":"Métro",
"bibliotheque":"Bibliothèque",
"Réunions avec les partenaires hospitaliers sur leur site":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"SITE CNRS":"Autres sites (CNRS, partenaires)",
"Bibliothèque, archives":"Bibliothèque",
"terrain" :"Terrains",
"bureau de collègues dans d'autres universités":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèque":"Bibliothèque",
"bibliothèque et archives":"Bibliothèque",
"bureau hors P8":"Autres sites (CNRS, partenaires)",
"bibliothèques":"Bibliothèque",
"Bureau MSH OU BIBLIOTHÈQUE MAIS JE MANQUE CRUELLEMENT D'UN ESPACE DE TRAVAIL":"Autres sites (Campus Condorcet, CNRS, partenaires)|Bibliothèque",
"Où je peux, puisque nous n'avon pas de bureaux. Lorsque je peux, j'évite le campus car : difficultés d'accès, de connexion ; risque que les salles soient déjà prises, etc. Cela est très bloquant pour tout travail d'équipe.":"Où je peux",
"partout":"Où je peux",
"Au laboratoire CEMTI":"Laboratoire",
"où je peux, je n'ai pas de bureau à la fac (15 m2 pour 6 titulaires)":"Où je peux",
"colloques, train, hotel":"Déplacement",
"dans des bibliothèque universitaires, au laboratoire":"Bibliothèque|Laboratoire",
"Transports en commun, salles d'attente...":"Où je peux",
"Lieux de recherche" :"Où je peux",
"Campus Condorcet (chercheur associé à l'Ined)":"Autres sites (Campus Condorcet, CNRS, partenaires)",
"Bibliothèques P8 et ailleurs":"Bibliothèque|Où je peux",
"Dans un bureau sur site partagé avec 5 autres collègues, trop petit, peu propice à la concentration et pas adapté aux réunions en visio":"Laboratoire",
"Bibliothèques.. Je trouve incongru la question du bureau dont nous ne disposons pas du tout (un peu plus depuis la construction de la Maison de la Recherche mais la chose n'a visiblement pas été pensée jusqu'au bout ou vraiment pour les enseignants chercheurs":"Bibliothèque"
}

In [89]:
df0["q26_1_autre_rec"] = df0.q26_1_autre_env.map(dict_autre_env.get)


,q26_1_autre_rec
0,None
1,None
2,None
3,None
4,None
...,...
114,None
115,Bibliothèque|Où je peux
116,None
117,None


In [96]:
# on enregistre les nouvelles variables dans le dataframe original
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "w") as file_out:
    df0.to_csv(file_out, sep=",", index= False)

KeyboardInterrupt: 

In [95]:
dic_new_var = {'name':'113. Autre, précisez',
               'label':'q26_1_autre_rec',
               'group':'13_env_travail',
               'personal_data':False,
               'type':'nominal_multiple',
               'opened_question':False,
               'type_panda':df0.q26_1_autre_rec.dtypes,
               'no_question':113,
               'question':"Où travaillez-vous le plus souvent? Si autre, précisez",
               'question_family':'q26',
               'comment':"recode les autre lieux"
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])
new_variable
df_col1 = pd.concat([df_col,new_variable], ignore_index = True)
df_col1

,name,label,group,personal_data,type,opened_question,type_panda,no_question,question,question_family,comment
0,N°Obs,qno_obs,NaN,False,numérique,False,int64,0.0,NaN,qno,NaN
1,1. so_principles,q1_so_principles,1_connaissance_so,False,ordinal,False,object,1.0,1/ Diriez-vous que vous êtes familier.ère des ...,q1,NaN
2,2. hal_depot,q2_hal_depot,1_connaissance_so,False,ordinal,False,object,2.0,2/ Avez-vous déjà déposé une production scient...,q2,NaN
3,3. octavi_depot,q3_octavi_depot,2_bib_num,False,nominal_multiple,False,object,3.0,3/ Avez-vous des productions déposées sur Octa...,q3,NaN
4,4. open_data,q4_diff_data,3_entrepot,False,nominal_multiple,False,object,4.0,4/ Avez-vous déjà diffusé vos données de reche...,q4,NaN
...,...,...,...,...,...,...,...,...,...,...,...
162,62. Logiciels de SIG (ex. QGis),q16_sig_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
163,63. Système de gestion de base de données,q16_sgbd_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
164,64. Autre,q16_autre_help,9_help_outil,False,ordinal,False,object,19.0,7/ Ressentez-vous le besoin d'un accompagnemen...,q16,NaN
165,65. other_accomp_software,q16_1_other_software_help,9_help_outil,False,texte_libre,True,object,20.0,"7.1/ Si Autre, précisez lequel (lesquels) ?",q16,NaN
